# Lab 3 — Delta Lake ACID Transactions, Schema Enforcement & Evolution

**Goal:** Inspect Delta history, test type/schema behavior, and evolve a table schema safely.

**Prerequisite:** Lab 1's `workspace.lab_db.customers` table. The first cell recreates a separate lab table so this exercise is repeatable without damaging Lab 1.

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.lab_db.customers_schema_lab (
  customer_id INT,
  name STRING,
  email STRING,
  signup_date DATE,
  status STRING
)
USING DELTA;

In [0]:
%sql
INSERT INTO workspace.lab_db.customers_schema_lab VALUES
  (101, 'Alice Smith', 'alice@example.com', DATE '2024-01-15', 'Active'),
  (102, 'Bob Jones', 'bob@example.com', DATE '2024-02-01', 'Pending');

In [0]:
%sql
DESCRIBE TABLE workspace.lab_db.customers_schema_lab;

In [0]:
%sql
DESCRIBE HISTORY workspace.lab_db.customers_schema_lab;

## Schema evolution using PySpark

This appends a new row with an additional `membership_tier` column. `mergeSchema` allows Delta to add the new column to the existing table schema.

In [0]:
from datetime import date

df_new = spark.createDataFrame(
    [(103, "Charlie Brown", "charlie@example.com", date(2024, 3, 10), "Active", "Tier-1")],
    ["customer_id", "name", "email", "signup_date", "status", "membership_tier"]
)

(df_new.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable("workspace.lab_db.customers_schema_lab"))

spark.sql("DESCRIBE TABLE workspace.lab_db.customers_schema_lab").show(truncate=False)
spark.table("workspace.lab_db.customers_schema_lab").orderBy("customer_id").show(truncate=False)

## Schema enforcement exercise

The next cell intentionally tries to append an incompatible schema without evolution. It is expected to fail; the exception is caught so the notebook can continue.

In [0]:
from pyspark.sql import functions as F

bad_df = spark.createDataFrame([(104, "Dana", "dana@example.com", "Active")],
                               ["customer_id", "name", "email", "status"])
try:
    (bad_df.write.format("delta").mode("append")
        .saveAsTable("workspace.lab_db.customers_schema_lab"))
    print("Write succeeded; inspect the schema and explain why it was accepted.")
except Exception as exc:
    print("Expected schema enforcement error:")
    print(str(exc)[:1200])

In [0]:
%sql
SELECT * FROM workspace.lab_db.customers_schema_lab ORDER BY customer_id;